In [53]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_selection import RFECV, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list


### feature selection step 1: ICC calculation for radimoics features from reader 1 and reader 2

In [55]:
df_reader1 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')
df_reader2 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized_reader2.xlsx')

# we only keep the rows in df_reader1 that are also in df_reader2 based on Patient_set and Patient_index
df_reader1 = df_reader1[df_reader1['Patient_index'].isin(df_reader2['Patient_index']) & df_reader1['Patient_set'].isin(df_reader2['Patient_set'])]
print(f'Number of cases in reader 1 after matching: {len(df_reader1)}')

non_feature_cols = ['Patient_set','Patient_index', 'Image_filepath', 'Mask_filepath']
feature_cols = [col for col in df_reader1.columns if col not in non_feature_cols]

# we need to calculate the ICC for each feature between reader 1 and reader 2, if it's >0.75, we keep it
# calculate ICC, import packages 

icc_rows = []
for f in feature_cols:
    x = df_reader1[f].values
    y = df_reader2[f].values
    icc = ff.icc2_1(x, y)
    if icc<0.75:
        print('feature:', f, ' ICC:', icc)
    icc_rows.append({'Feature': f, 'ICC': icc})

# only keep features with ICC > 0.75
icc_df = pd.DataFrame(icc_rows)
selected_features = icc_df[icc_df['ICC'] > 0.75]['Feature'].tolist()
print('original number of features:', len(feature_cols))
print(f'Number of features with ICC > 0.75: {len(selected_features)}')

# dropped features
dropped_features = icc_df[icc_df['ICC'] <= 0.75]['Feature'].tolist()
# save dropped features to excel, file name: dropped_features.xlsx, sheet_name: 'inter_reader_icc'
dropped_df = pd.DataFrame({'dropped_feature': dropped_features})

with pd.ExcelWriter('/host/d/projects/Habitats/radiomics/whole_image/dropped_features.xlsx', engine='openpyxl') as writer:
    dropped_df.to_excel(writer, sheet_name='inter_reader_icc', index=False)

# now we create "df" for reader 1 with only selected features
df_reader1 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')
df_reader1_selected = df_reader1[['Patient_set','Patient_index', 'Image_filepath', 'Mask_filepath'] + selected_features]
print('Shape of df_reader1_selected:', df_reader1_selected.shape)


Number of cases in reader 1 after matching: 28
feature: wavelet-LLH_glszm_GrayLevelNonUniformityNormalized  ICC: 0.6263630706816321
feature: wavelet-LLH_glszm_LargeAreaEmphasis  ICC: 0.5329332740187652
feature: wavelet-LLH_glszm_LargeAreaLowGrayLevelEmphasis  ICC: 0.3768460239335729
feature: wavelet-LLH_glszm_ZoneEntropy  ICC: 0.7283365041657534
feature: wavelet-LLH_glszm_ZoneVariance  ICC: 0.543677394394188
feature: wavelet-LLH_ngtdm_Busyness  ICC: 0.4740578781919682
feature: wavelet-LHH_glrlm_LongRunLowGrayLevelEmphasis  ICC: 0.6917611185313896
feature: wavelet-LHH_glszm_LargeAreaLowGrayLevelEmphasis  ICC: 0.5612562639490835
feature: wavelet-LHH_ngtdm_Busyness  ICC: 0.31184178856412614
feature: wavelet-HLH_glcm_ClusterShade  ICC: -0.08047851071601521
feature: wavelet-HLH_ngtdm_Busyness  ICC: 0.5524725353820513
original number of features: 1106
Number of features with ICC > 0.75: 1095
Shape of df_reader1_selected: (330, 1099)


### feature selection step 2: PCC


In [56]:
df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')

non_feature_cols = ["Patient_set","Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]

X = df[feature_cols].copy()

corr = X.corr(method='pearson').abs()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# 5) 找到需要删除的列：如果该列与任何其他列相关性 > 阈值，就删它
threshold = 0.90
to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]

print(f"Total features: {len(feature_cols)}")
print(f"Dropped due to PCC > {threshold}: {len(to_drop)}")
print(f"Remaining: {len(feature_cols) - len(to_drop)}")


Total features: 1106
Dropped due to PCC > 0.9: 821
Remaining: 285


In [57]:
X_selected = X.drop(columns=to_drop)

df_pcc = pd.concat([df[non_feature_cols], X_selected], axis=1)
df_pcc.to_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_PCC.xlsx', index=False)

# save dropped features to excel, file name: dropped_features.xlsx, sheet_name: 'pcc'
dropped_df = pd.DataFrame({'dropped_feature': to_drop})
with pd.ExcelWriter('/host/d/projects/Habitats/radiomics/whole_image/dropped_features.xlsx', engine='openpyxl', mode='a') as writer:
    dropped_df.to_excel(writer, sheet_name='pcc', index=False)
